### **Transformers y atención como bloque universal**

Este cuaderno conserva indica cuatro aspectos de ingeniería y precisión conceptual:

1. **Una estrategia posicional canónica por baseline.** El modelo admite `pos_mode=None`, `"sin"` o `"learned"`. El baseline recomendado usa **RoPE solo** (`pos_mode=None, use_rope=True`). Las combinaciones se permiten únicamente como *ablaciones* explícitas.
2. **RoPE cacheado.** Los valores seno/coseno se precomputan como *buffers* hasta `max_len` y se reutilizan en cada `forward`.
3. **ALiBi con pendientes oficiales.** Se usa el algoritmo `get_slopes` del código oficial de ALiBi, incluyendo el caso de un número de heads que no sea potencia de dos.
4. **Evaluación desde la primera práctica.** Además de construir atención, se verifican invariantes: normalización de softmax y ausencia de fuga hacia tokens futuros.

> Nota pedagógica: combinar PE absoluta + RoPE, RoPE + ALiBi u otras variantes no es matemáticamente imposible. Sin embargo, no debe aparecer como configuración por defecto. En este cuaderno esas combinaciones requieren `allow_hybrid_positional=True` y se tratan como experimentos de ablación.

#### **1. Preparación y reproducibilidad**

El cuaderno es CPU-first y no necesita GPU, APIs externas ni `torchtext`.

In [ ]:
import math
import random
import warnings
from collections import Counter
from typing import List, Optional

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

#### **2. Texto -> tokens -> ids -> embeddings**

Se usa tokenización por espacios solo para aislar la mecánica del Transformer. En un sistema real se usaría un tokenizador subword entrenado o asociado al modelo.

In [ ]:
corpus = [
    "the llama learns quickly",
    "the llama runs fast",
    "the dog runs fast",
    "the dog barks loudly",
    "the horse runs fast",
    "the horse eats hay",
    "the llama eats hay",
    "attention is a universal block",
    "transformers use self attention",
    "decoder only models predict next token",
    "encoder decoder models use cross attention",
]

def basic_tokenize(text: str) -> List[str]:
    return text.lower().strip().split()

class SimpleVocab:
    def __init__(self, token_lists: List[List[str]], min_freq: int = 1):
        specials = ["<pad>", "<bos>", "<eos>", "<unk>"]
        counter = Counter(tok for toks in token_lists for tok in toks)
        self.itos = specials[:]
        for tok, freq in counter.items():
            if freq >= min_freq and tok not in specials:
                self.itos.append(tok)
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}
        self.pad_id = self.stoi["<pad>"]
        self.bos_id = self.stoi["<bos>"]
        self.eos_id = self.stoi["<eos>"]
        self.unk_id = self.stoi["<unk>"]

    def encode(self, text: str, add_bos: bool = True, add_eos: bool = True) -> List[int]:
        ids = [self.bos_id] if add_bos else []
        ids.extend(self.stoi.get(tok, self.unk_id) for tok in basic_tokenize(text))
        if add_eos:
            ids.append(self.eos_id)
        return ids

    def decode(self, ids: List[int]) -> str:
        ignored = {"<pad>", "<bos>", "<eos>"}
        return " ".join(self.itos[i] for i in ids if self.itos[i] not in ignored)

token_lists = [basic_tokenize(x) for x in corpus]
vocab = SimpleVocab(token_lists)
vocab_size = len(vocab.itos)

sample = "the llama runs fast"
ids = vocab.encode(sample)
print("Vocabulario:", vocab_size)
print("Texto:", sample)
print("IDs:", ids)
print("Reconstrucción:", vocab.decode(ids))

#### **Actividad guiada**

1. Explica qué información cambia en cada etapa: `texto -> tokens -> ids -> embeddings`.
2. ¿Qué limitación introduce la tokenización por espacios?
3. ¿Qué problema resuelven BPE, WordPiece o SentencePiece?
4. Identifica qué parte de este pipeline pertenece al modelo y qué parte pertenece al sistema que lo prepara y ejecuta.

#### **3. Estrategias de posición**

Distinguiremos cuatro familias:

- **PE sinusoidal absoluta:** se suma a los embeddings.
- **PE aprendida absoluta:** se suma a los embeddings y se entrena.
- **RoPE:** rota pares de dimensiones de Q y K dentro de la atención.
- **ALiBi:** no suma embeddings posicionales; añade un sesgo lineal dependiente de distancia a los logits de atención.

Para un baseline didáctico se emplea **una** estrategia. Las combinaciones quedan para ablaciones explícitas.

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32)
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, :x.size(1), :]

class LearnedPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        self.pos_emb = nn.Embedding(max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, _ = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        return x + self.pos_emb(pos)

#### **3.1 RoPE con cache precomputado**

El material anterior reconstruía `cos` y `sin` en cada `forward`. Eso produce el resultado correcto, pero hace trabajo repetido. Aquí se precomputan hasta `max_len` y se almacenan como *buffers* del módulo.

In [ ]:
def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., ::2]
    x2 = x[..., 1::2]
    out = torch.stack((-x2, x1), dim=-1)
    return out.flatten(-2)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_len: int = 512, base: float = 10000.0):
        super().__init__()
        if dim % 2 != 0:
            raise ValueError("RoPE requiere dimensión par por head")
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        positions = torch.arange(max_len, dtype=torch.float32)
        freqs = torch.outer(positions, inv_freq)
        angles = torch.repeat_interleave(freqs, 2, dim=-1)
        self.register_buffer("cos_cached", angles.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin_cached", angles.sin()[None, None, :, :], persistent=False)
        self.max_len = max_len

    def forward(self, q: torch.Tensor, k: torch.Tensor):
        T = q.size(-2)
        if T > self.max_len:
            raise ValueError(f"T={T} excede max_len={self.max_len}; reconstruya el módulo con mayor max_len")
        cos = self.cos_cached[:, :, :T, :].to(dtype=q.dtype)
        sin = self.sin_cached[:, :, :T, :].to(dtype=q.dtype)
        q = q * cos + rotate_half(q) * sin
        k = k * cos + rotate_half(k) * sin
        return q, k

# Comprobación: el buffer se reutiliza, no se reconstruye en cada forward.
rope_demo = RotaryEmbedding(dim=8, max_len=32)
print(rope_demo.cos_cached.shape, rope_demo.sin_cached.shape)

#### **3.2 ALiBi con pendientes del algoritmo oficial**

Para `n` heads potencia de dos, las pendientes forman una progresión geométrica. El código oficial también especifica cómo construirlas cuando `n` no es potencia de dos. ALiBi se añade a los logits antes de softmax; la máscara causal se mantiene como mecanismo separado.

In [ ]:
def get_alibi_slopes(n_heads: int) -> torch.Tensor:
    """Pendientes según el algoritmo oficial de ALiBi."""
    def slopes_power_of_2(n: int):
        start = 2 ** (-(2 ** -(math.log2(n) - 3)))
        ratio = start
        return [start * (ratio ** i) for i in range(n)]

    if math.log2(n_heads).is_integer():
        values = slopes_power_of_2(n_heads)
    else:
        closest = 2 ** math.floor(math.log2(n_heads))
        values = slopes_power_of_2(closest)
        extra = get_alibi_slopes(2 * closest).tolist()[0::2][:n_heads - closest]
        values += extra
    return torch.tensor(values, dtype=torch.float32)

def build_alibi_bias(num_heads: int, seq_len: int, device=None) -> torch.Tensor:
    slopes = get_alibi_slopes(num_heads).to(device=device)  # [H]
    q_pos = torch.arange(seq_len, device=device).view(seq_len, 1)
    k_pos = torch.arange(seq_len, device=device).view(1, seq_len)
    past_distance = (q_pos - k_pos).clamp(min=0).float()      # [T,T]
    bias = -slopes[:, None, None] * past_distance[None, :, :] # [H,T,T]
    return bias.unsqueeze(0)                                  # [1,H,T,T]

print("Pendientes H=8:", get_alibi_slopes(8).tolist())

#### Actividad guiada - posición como hipótesis experimental

Compara, manteniendo constantes datos, semilla, arquitectura y presupuesto de entrenamiento:

| Variante | `pos_mode` | `use_rope` | `use_alibi` | Tipo |
|---|---|---:|---:|---|
| A | `"sin"` | False | False | baseline clásico |
| B | `"learned"` | False | False | baseline aprendido |
| C | `None` | True | False | **baseline moderno recomendado** |
| D | `None` | False | True | ALiBi |
| E | `"sin"` | True | False | **ablación híbrida**, no baseline |

La variante E debe ejecutarse solo con `allow_hybrid_positional=True`. El objetivo es comprobar empíricamente si la combinación aporta o perjudica en este problema pequeño; no presentarla como práctica estándar.

#### **4. Atención producto punto escalado y causalidad**

In [ ]:
def make_causal_mask(T: int, device=None) -> torch.Tensor:
    return torch.tril(torch.ones(T, T, dtype=torch.bool, device=device))

def scaled_dot_product_attention(q, k, v, causal=True, extra_bias=None):
    # q,k,v: [B,H,T,Dh]
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))
    if extra_bias is not None:
        scores = scores + extra_bias
    if causal:
        mask = make_causal_mask(q.size(-2), q.device)
        scores = scores.masked_fill(~mask[None, None, :, :], float("-inf"))
    attn = torch.softmax(scores, dim=-1)
    out = attn @ v
    return out, attn

B, H, T, Dh = 1, 2, 6, 8
q = torch.randn(B, H, T, Dh)
k = torch.randn(B, H, T, Dh)
v = torch.randn(B, H, T, Dh)
out, attn = scaled_dot_product_attention(q, k, v, causal=True)
print("out:", out.shape, "attn:", attn.shape)

#### **4.1 EVALUATE: invariantes mínimos**

Un laboratorio de sistemas no debería quedarse en "código corre". Verificamos propiedades observables.

In [ ]:
def attention_invariants(attn: torch.Tensor):
    T = attn.size(-1)
    row_sum_error = (attn.sum(dim=-1) - 1.0).abs().max().item()
    future = torch.triu(torch.ones(T, T, dtype=torch.bool, device=attn.device), diagonal=1)
    future_values = attn.masked_select(future[None, None, :, :])
    future_leakage = future_values.max().item() if future_values.numel() else 0.0
    return {
        "normalization_error": row_sum_error,
        "future_attention_leakage": future_leakage,
    }

metrics = attention_invariants(attn)
print(metrics)
assert metrics["normalization_error"] < 1e-5
assert metrics["future_attention_leakage"] < 1e-7

#### **5. Multi-head attention (atención multihead)**

El módulo mantiene una separación clara entre:

- PE aditiva en la entrada (`pos_mode`),
- RoPE aplicado a Q/K,
- ALiBi como bias de logits.

Por defecto se rechaza activar más de una estrategia posicional. Para una ablación híbrida debe utilizarse `allow_hybrid_positional=True`.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, max_len: int = 128,
                 dropout: float = 0.0, use_rope: bool = False,
                 use_alibi: bool = False):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model debe ser divisible por num_heads")
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.use_rope = use_rope
        self.use_alibi = use_alibi
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.head_dim, max_len=max_len) if use_rope else None
        if use_alibi:
            self.register_buffer("alibi_cache", build_alibi_bias(num_heads, max_len), persistent=False)
        else:
            self.alibi_cache = None

    def forward(self, x: torch.Tensor):
        B, T, D = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q = q.transpose(1, 2)  # [B,H,T,Dh]
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        if self.rope is not None:
            q, k = self.rope(q, k)

        bias = None
        if self.alibi_cache is not None:
            bias = self.alibi_cache[:, :, :T, :T].to(device=x.device, dtype=q.dtype)

        out, attn = scaled_dot_product_attention(q, k, v, causal=True, extra_bias=bias)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.out_proj(self.dropout(out)), attn

#### **6. Bloque Transformer decoder-only**

Se usa **pre-norm**: `LayerNorm -> attention -> residual -> LayerNorm -> FFN -> residual`.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, max_len: int,
                 dropout: float, use_rope: bool, use_alibi: bool):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, max_len, dropout, use_rope, use_alibi)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        attn_out, attn = self.attn(self.norm1(x))
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x, attn

#### **7. Tiny decoder-only LM con configuración posicional explícita**

In [ ]:
def validate_positional_config(pos_mode, use_rope, use_alibi, allow_hybrid_positional=False):
    if pos_mode not in {None, "sin", "learned"}:
        raise ValueError("pos_mode debe ser None, 'sin' o 'learned'")
    active = int(pos_mode is not None) + int(use_rope) + int(use_alibi)
    if active == 0:
        warnings.warn("Modelo sin información posicional explícita: útil solo como control experimental.")
    if active > 1 and not allow_hybrid_positional:
        raise ValueError(
            "Hay más de una estrategia posicional activa. Use una sola para el baseline "
            "o establezca allow_hybrid_positional=True para una ablación explícita."
        )

class TinyDecoderOnlyLM(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 64, num_heads: int = 4,
                 n_layers: int = 2, max_len: int = 128, dropout: float = 0.1,
                 pos_mode: Optional[str] = None, use_rope: bool = True,
                 use_alibi: bool = False, allow_hybrid_positional: bool = False):
        super().__init__()
        validate_positional_config(pos_mode, use_rope, use_alibi, allow_hybrid_positional)
        self.max_len = max_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        if pos_mode == "sin":
            self.pos_enc = SinusoidalPositionalEncoding(d_model, max_len)
        elif pos_mode == "learned":
            self.pos_enc = LearnedPositionalEncoding(d_model, max_len)
        else:
            self.pos_enc = nn.Identity()

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, max_len, dropout, use_rope, use_alibi)
            for _ in range(n_layers)
        ])
        self.norm_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        if idx.size(1) > self.max_len:
            raise ValueError("Secuencia mayor que max_len")
        x = self.pos_enc(self.token_emb(idx))
        attentions = []
        for block in self.blocks:
            x, attn = block(x)
            attentions.append(attn)
        logits = self.lm_head(self.norm_f(x))
        return logits, attentions

# Baseline canónico: RoPE solo.
model_test = TinyDecoderOnlyLM(vocab_size=vocab_size, pos_mode=None,
                               use_rope=True, use_alibi=False).to(device)
example = torch.tensor([vocab.encode("the llama runs fast")], device=device)
logits, attentions = model_test(example)
print("logits:", logits.shape, "capas de atención:", len(attentions))

#### **7.1 Comprobación de que la combinación híbrida ya no ocurre accidentalmente**

In [ ]:
try:
    TinyDecoderOnlyLM(vocab_size=vocab_size, pos_mode="sin", use_rope=True)
except ValueError as e:
    print("Configuración rechazada correctamente:")
    print(e)

# Solo para una ablación declarada:
hybrid_model = TinyDecoderOnlyLM(
    vocab_size=vocab_size,
    pos_mode="sin",
    use_rope=True,
    allow_hybrid_positional=True,
)
print("ablación híbrida creada de forma explícita.")

#### **8. Mini-entrenamiento autoregresivo**

El objetivo es observar el flujo completo, no obtener un buen modelo de lenguaje. En entrenamiento causal se predicen todos los siguientes tokens en paralelo mediante secuencias desplazadas; en inferencia la generación sigue siendo autoregresiva.

In [ ]:
sequences = [torch.tensor(vocab.encode(s), dtype=torch.long) for s in corpus]
max_seq = max(len(s) for s in sequences)

def pad(seq, length):
    if len(seq) >= length:
        return seq
    return torch.cat([seq, torch.full((length-len(seq),), vocab.pad_id, dtype=torch.long)])

data = torch.stack([pad(s, max_seq) for s in sequences]).to(device)
inputs = data[:, :-1]
targets = data[:, 1:]

model = TinyDecoderOnlyLM(
    vocab_size=vocab_size,
    d_model=48,
    num_heads=4,
    n_layers=2,
    max_len=max_seq,
    dropout=0.0,
    pos_mode=None,
    use_rope=True,
    use_alibi=False,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
losses = []
for epoch in range(12):
    optimizer.zero_grad()
    logits, _ = model(inputs)
    loss = F.cross_entropy(
        logits.reshape(-1, vocab_size),
        targets.reshape(-1),
        ignore_index=vocab.pad_id,
    )
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    losses.append(loss.item())

print("Loss inicial/final:", round(losses[0], 4), "->", round(losses[-1], 4))
plt.plot(losses)
plt.xlabel("época")
plt.ylabel("cross-entropy")
plt.title("Mini-entrenamiento - RoPE only")
plt.show()

#### **9. Decodificación mínima**

La decodificación se deja como puente a la Semana 2. Aquí solo se muestra greedy decoding para no mezclar todavía temperatura, top-k y top-p con los objetivos de la Semana 1.

In [ ]:
@torch.no_grad()
def generate_greedy(model, prefix: str, max_new_tokens: int = 6):
    ids = vocab.encode(prefix, add_eos=False)
    x = torch.tensor([ids], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        if x.size(1) >= model.max_len:
            break
        logits, _ = model(x)
        next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        x = torch.cat([x, next_id], dim=1)
        if next_id.item() == vocab.eos_id:
            break
    return vocab.decode(x[0].tolist())

print(generate_greedy(model, "the llama", max_new_tokens=4))

#### **10. Actividad experimental de cierre**

Diseña y ejecuta una comparación controlada entre **tres** variantes posicionales. Como mínimo:

1. mantén igual semilla, datos, arquitectura y número de actualizaciones,
2. registra `loss` final y curva de entrenamiento,
3. comprueba `normalization_error` y `future_attention_leakage`,
4. describe una limitación del experimento,
5. explica por qué un resultado en este corpus diminuto **no** permite concluir cuál estrategia es mejor para LLM de gran escala.

Formato de entrega:

`hipótesis -> baseline -> modificación -> métrica -> resultado -> análisis de error -> conclusión`

#### **11. Lectura de implementación real**

Para conectar el cuaderno con software de referencia:

- `annotated_deep_learning_paper_implementations/labml_nn/transformers/mha.py`: ubique Q, K, V, `QK^T`, escala `1/sqrt(d_k)`, máscara, softmax y multiplicación por V.
- `annotated_deep_learning_paper_implementations/labml_nn/transformers/models.py`: ubique `LayerNorm`, residuals, FFN y composición de `TransformerLayer`.
- `annotated_deep_learning_paper_implementations/labml_nn/transformers/rope/__init__.py`: observa que la implementación mantiene un cache de seno/coseno y solo lo reconstruye si la secuencia excede el cache existente.

No se pide copiar estas implementaciones. Se pide **leerlas y defender cómo la ecuación se transforma en software**.